In [1]:
import numpy as np
import matplotlib.pyplot as plt
import aopy
import os
import pandas as pds
from db import dbfunctions as db
from ipywidgets import interactive, widgets
import scipy
import h5py
from tqdm.auto import tqdm 
import seaborn as sn
import sklearn
from sklearn.decomposition import PCA
from itertools import compress
# import multiprocessing as mp
import time
import math
# from scipy.fft import fft
import glob
from datetime import date

/home/aolab/miniconda3/envs/np_targeting/lib/python3.9/site-packages/one/alf/files.py:10: FutureWarning: `one.alf.files` will be removed in version 3.0. Use `one.alf.path` instead.
  warnings.warn(


In [2]:
from local_functions import load_yaml_params, get_pseudopopulation_raster
subject = 'affi'
param_filename = 'spatiotemporal_structure_params.yaml'
ds_params, data_paths, save_paths, filenames, postproc_params, uq_params, analysis_params, vis_params, target_colors = load_yaml_params(param_filename)
align_event = analysis_params['align_events'][0]

In [3]:
ecog_dec_acc = aopy.data.base.load_hdf_group(save_paths['postproc_data'], filenames['ecog_interp'])
day_colors = ecog_dec_acc[subject]['day_colors']

In [4]:
lda_results, fr_zscore, lda_metadata = aopy.data.pkl_read(f"{subject}_{filenames['lda_result']}", save_paths['postproc_data'])

In [ ]:
start = time.time()
aopy.utils.release_memory_limit()
df, rasters, preproc_metadata = aopy.data.base.pkl_read(f"{subject}_{filenames['preprocessed']}", save_paths['postproc_data'])
print(f"{np.round((time.time()-start)/60)} min to load preprocessed data")
nrecs = preproc_metadata['nrecs']
recording_site = preproc_metadata['recording_sites'] # will be the same for all align events
implants = ['NPinsert72' if preproc_metadata['implant'][irec] == 'NP_Insert72' else 'NPinsert137' for irec in range(len(preproc_metadata['implant']))] #Rename because name in bmi3d is slightly different (TODO)
dates = np.unique(df['date'])
ntargets = len(np.unique(df['target_idx']))

In [ ]:
neural_idx_start = lda_metadata['trelevant1_smooth']
neural_idx_stop = lda_metadata['trelevant2_smooth']
kin_idx_start = lda_metadata['trelevant1']
kin_idx_stop = lda_metadata['trelevant2']

In [ ]:
data_dict = {'trial': [], 'unit': [], 'subject': [], 'rec_site': [], 'rec_rcaxis': [], 'rec_mlaxis': [], 'penetration': [], 'firing_rate': [], 'target': [], 'cursor_traj': [], 'cursor_vel': []}
# unit_info = {'rec_number': [], 'date': [], 'rec_site': [], 'rec_xpos': [], 'rec_ypos': [], 'implant': [],
#              'unit_label': [], 'unit_idx': []}
nt, _, ntrials = fr_zscore[align_event]['align_spikes_zscore_smooth'][0].shape
npenetrations = preproc_metadata['nrecs']
for ip in tqdm(range(npenetrations)):
    # Only include sites without nans
    if ~np.isnan(fr_zscore[align_event]['align_spikes_zscore_smooth'][ip]).all():
        trial_mask = np.array(df[df['penetration']==ip]['good_trial'])
        cursor_traj_dwns_temp = aopy.precondition.downsample(rasters['cursor_pos'][align_event][ip][:,trial_mask,:], postproc_params['exp_samplerate'], int(1/postproc_params['bin_width']))
        cursor_vel_dwns_temp = aopy.precondition.downsample(rasters['cursor_velo'][align_event][ip][:,trial_mask,:], postproc_params['exp_samplerate'], int(1/postproc_params['bin_width']))
        for itrial in range(ntrials):
            nunits = fr_zscore[align_event]['align_spikes_zscore_smooth'][ip].shape[1]
            for iunit in range(nunits):
                data_dict['penetration'].append(ip)
                data_dict['trial'].append(itrial)
                data_dict['unit'].append(iunit)
                data_dict['subject'].append(subject)
                data_dict['rec_site'].append(recording_site[ip])
                data_dict['rec_rcaxis'].append(ecog_dec_acc[subject]['rc_axis'][ip,0])
                data_dict['rec_mlaxis'].append(ecog_dec_acc[subject]['rc_axis'][ip,1])

                data_dict['firing_rate'].append(fr_zscore[align_event]['align_spikes_zscore_smooth'][ip][neural_idx_start:neural_idx_stop,iunit,itrial])
                data_dict['target'].append(fr_zscore[align_event]['target'][ip][itrial])

                # Cursor trajectory and velocity need to be downsampled
                data_dict['cursor_traj'].append(cursor_traj_dwns_temp[kin_idx_start:kin_idx_stop,itrial,:])
                data_dict['cursor_vel'].append(cursor_vel_dwns_temp[kin_idx_start:kin_idx_stop,itrial,:])


In [ ]:
data_df = pds.DataFrame(data_dict)

In [ ]:
# Save data
aopy.data.base.pkl_write(f"{subject}_{filenames['postproc']}", data_df, save_paths['postproc_data'])

In [ ]:
# subject = 'beignet'
filepath = '/media/moor-data/results/Ryan/spatiotemporal_structure/np_analysis_preproc_data'
test_df = aopy.data.pkl_read(f"{subject}_np_postproc", filepath)

In [ ]:
test_df.head()

In [ ]:
# Save dataframe